# Tests for ExecutionManager with SingleWorkerPool (Async/Main Pool)

SingleWorkerPool runs tasks in the main process using asyncio, which is useful for
I/O-bound tasks and when you don't need process isolation.

In [ ]:
#|default_exp execution_manager.test_execution_manager_async

In [ ]:
#|export
import pytest
import asyncio
from datetime import datetime

from netrun.pool.aio import SingleWorkerPool

from netrun.execution_manager import (
    ExecutionManager,
    RunAllocationMethod,
)

# Import worker functions from the workers module
from tests.execution_manager.workers import (
    add_numbers,
    multiply_numbers,
    function_with_print,
    slow_function,
    function_returns_non_serializable,
    function_with_kwargs,
    async_add,
    function_with_multiple_prints,
    slow_printing_function,
)

## Test Starting and Closing

In [ ]:
#|export
@pytest.mark.asyncio
async def test_start_and_close():
    """Test starting and closing the manager with SingleWorkerPool."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    await manager.start()
    assert manager._started is True
    assert "pool" in manager._pools
    await manager.close()

In [ ]:
await test_start_and_close();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_context_manager():
    """Test using ExecutionManager as async context manager."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        assert manager._started is True

    # After exit, pools should be closed

In [ ]:
await test_context_manager();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_immediate_close():
    """Test that immediate close after start doesn't raise errors."""
    # Run multiple times to catch race conditions
    for _ in range(10):
        manager = ExecutionManager({
            "pool": (SingleWorkerPool, {}),
        })
        async with manager:
            pass  # Immediately close without doing anything

In [ ]:
await test_immediate_close();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_double_start_raises():
    """Test that starting twice raises an error."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    await manager.start()
    try:
        with pytest.raises(RuntimeError, match="already started"):
            await manager.start()
    finally:
        await manager.close()

In [ ]:
await test_double_start_raises();

## Test pool_ids and get_num_workers

In [ ]:
#|export
@pytest.mark.asyncio
async def test_pool_ids():
    """Test getting pool IDs."""
    manager = ExecutionManager({
        "pool_a": (SingleWorkerPool, {}),
        "pool_b": (SingleWorkerPool, {}),
    })

    async with manager:
        pool_ids = [pool_id for pool_id, _ in manager.pools]
        assert "pool_a" in pool_ids
        assert "pool_b" in pool_ids
        assert len(pool_ids) == 2

In [ ]:
await test_pool_ids();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_get_num_workers():
    """Test getting number of workers in a pool."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        # SingleWorkerPool always has 1 worker
        assert manager.get_num_workers("pool") == 1

In [ ]:
await test_get_num_workers();

## Test send_function and run

In [ ]:
#|export
@pytest.mark.asyncio
async def test_send_function_and_run():
    """Test sending a function and running it."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        # Send the function to the worker
        await manager.send_function("pool", 0, "add", add_numbers)

        # Run the function
        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="add",
            send_channel=False,
            func_args=(3, 4),
            func_kwargs={},
        )

        assert result.result == 7
        assert result.pool_id == "pool"
        assert result.worker_id == 0
        assert result.converted_to_str is False

In [ ]:
await test_send_function_and_run();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_send_function_to_pool():
    """Test sending a function to all workers in a pool."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        # Send the function to all workers (just 1 for SingleWorkerPool)
        await manager.send_function_to_pool("pool", "multiply", multiply_numbers)

        # Run on the single worker
        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="multiply",
            send_channel=False,
            func_args=(5, 10),
            func_kwargs={},
        )

        assert result.result == 50

In [ ]:
await test_send_function_to_pool();

## Test JobResult

In [ ]:
#|export
@pytest.mark.asyncio
async def test_job_result_timestamps():
    """Test that JobResult has correct timestamps."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "slow", slow_function)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="slow",
            send_channel=False,
            func_args=(0.1,),
            func_kwargs={},
        )

        # Check timestamps are in correct order
        assert result.timestamp_utc_submitted <= result.timestamp_utc_started
        assert result.timestamp_utc_started <= result.timestamp_utc_completed

        # Check result
        assert result.result == "done"

In [ ]:
await test_job_result_timestamps();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_non_serializable_result():
    """Test that non-serializable results work in SingleWorkerPool (same process)."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "nonserialized", function_returns_non_serializable)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="nonserialized",
            send_channel=False,
            func_args=(),
            func_kwargs={},
        )

        # SingleWorkerPool runs in the same process, so non-serializable results work
        assert result.converted_to_str is False
        # The result should be a lambda function
        assert callable(result.result)
        assert result.result(5) == 5

In [ ]:
await test_non_serializable_result();

## Test Function with kwargs

In [ ]:
#|export
@pytest.mark.asyncio
async def test_function_with_kwargs():
    """Test running a function with keyword arguments."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "kwargs_fn", function_with_kwargs)

        # Test with only positional arg
        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="kwargs_fn",
            send_channel=False,
            func_args=(1,),
            func_kwargs={},
        )
        assert result.result == 111  # 1 + 10 + 100

        # Test with kwargs
        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="kwargs_fn",
            send_channel=False,
            func_args=(5,),
            func_kwargs={"b": 20, "c": 200},
        )
        assert result.result == 225  # 5 + 20 + 200

In [ ]:
await test_function_with_kwargs();

## Test Allocation Methods

In [ ]:
#|export
@pytest.mark.asyncio
async def test_round_robin_allocation():
    """Test round-robin job allocation with single worker."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function_to_pool("pool", "add", add_numbers)

        # Run 3 jobs sequentially with round-robin
        worker_ids = []
        for i in range(3):
            result = await manager.run_allocate(
                pool_worker_ids=["pool"],
                allocation_method=RunAllocationMethod.ROUND_ROBIN,
                func_import_path_or_key="add",
                send_channel=False,
                func_args=(i, 1),
                func_kwargs={},
            )
            worker_ids.append(result.worker_id)

        # With only 1 worker, we should always see worker 0
        assert worker_ids == [0, 0, 0]

In [ ]:
await test_round_robin_allocation();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_empty_workers_raises():
    """Test that empty worker list raises error."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "add", add_numbers)

        with pytest.raises(ValueError, match="No workers available"):
            await manager.run_allocate(
                pool_worker_ids=[],
                allocation_method=RunAllocationMethod.ROUND_ROBIN,
                func_import_path_or_key="add",
                send_channel=False,
                func_args=(1, 2),
                func_kwargs={},
            )

In [ ]:
await test_empty_workers_raises();

## Test get_worker_jobs

In [ ]:
#|export
@pytest.mark.asyncio
async def test_get_worker_jobs_empty():
    """Test get_worker_jobs when no jobs are running."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        jobs = manager.get_worker_jobs("pool", 0)
        assert jobs == []

In [ ]:
await test_get_worker_jobs_empty();

## Test Multiple Pools

In [ ]:
#|export
@pytest.mark.asyncio
async def test_multiple_pools():
    """Test running jobs on multiple SingleWorkerPools."""
    manager = ExecutionManager({
        "pool_a": (SingleWorkerPool, {}),
        "pool_b": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function_to_pool("pool_a", "add", add_numbers)
        await manager.send_function_to_pool("pool_b", "multiply", multiply_numbers)

        # Run on pool_a
        result1 = await manager.run(
            pool_id="pool_a",
            worker_id=0,
            func_import_path_or_key="add",
            send_channel=False,
            func_args=(5, 3),
            func_kwargs={},
        )

        # Run on pool_b
        result2 = await manager.run(
            pool_id="pool_b",
            worker_id=0,
            func_import_path_or_key="multiply",
            send_channel=False,
            func_args=(4, 7),
            func_kwargs={},
        )

        assert result1.result == 8
        assert result1.pool_id == "pool_a"
        assert result2.result == 28
        assert result2.pool_id == "pool_b"

In [ ]:
await test_multiple_pools();

## Test Async Functions

In [ ]:
#|export
@pytest.mark.asyncio
async def test_async_function():
    """Test running an async function."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "async_add", async_add)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="async_add",
            send_channel=False,
            func_args=(10, 20),
            func_kwargs={},
        )

        assert result.result == 30

In [ ]:
await test_async_function();

## Test Print Buffer

In [ ]:
#|export
@pytest.mark.asyncio
async def test_print_buffer_in_result():
    """Test that JobResult contains the print buffer."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "print_fn", function_with_print)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="print_fn",
            send_channel=False,
            func_args=("World",),
            func_kwargs={},
        )

        assert result.result == "greeted World"
        # Check print buffer contains the captured print
        assert len(result.print_buffer) == 1
        timestamp, text = result.print_buffer[0]
        assert "Hello, World!" in text
        assert isinstance(timestamp, datetime)

In [ ]:
await test_print_buffer_in_result();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_print_buffer_multiple_prints():
    """Test that JobResult captures multiple print statements."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "multi_print", function_with_multiple_prints)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="multi_print",
            send_channel=False,
            func_args=(5,),
            func_kwargs={},
        )

        assert result.result == "printed 5 lines"
        assert len(result.print_buffer) == 5
        for i, (timestamp, text) in enumerate(result.print_buffer):
            assert f"Line {i}" in text

In [ ]:
await test_print_buffer_multiple_prints();

In [ ]:
#|export
@pytest.mark.asyncio
async def test_on_print_callback():
    """Test the on_print callback receives print buffers during execution."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {"print_flush_interval": 0.05}),
    })

    received_buffers: list[list[tuple[datetime, str]]] = []

    def on_print(buffer):
        received_buffers.append(buffer)

    async with manager:
        await manager.send_function("pool", 0, "slow_print", slow_printing_function)

        # Run function that prints 5 times with delays
        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="slow_print",
            send_channel=False,
            func_args=(5, 0.08),  # 5 prints, 80ms delay each
            func_kwargs={},
            on_print=on_print,
        )

        assert result.result == "done"
        # All prints should be in the final result
        assert len(result.print_buffer) == 5

        # The on_print callback should have been called at least once
        # (since we have 5 prints with 80ms delay and 50ms flush interval)
        assert len(received_buffers) >= 1

        # All received buffers combined should equal the final print_buffer
        all_received = []
        for buf in received_buffers:
            all_received.extend(buf)
        assert len(all_received) == len(result.print_buffer)

In [ ]:
await test_on_print_callback();

## Test using SingleWorkerPool class directly

In [ ]:
#|export
@pytest.mark.asyncio
async def test_pool_class_directly():
    """Test using SingleWorkerPool class directly (not a string alias)."""
    manager = ExecutionManager({
        "pool": (SingleWorkerPool, {}),
    })

    async with manager:
        await manager.send_function("pool", 0, "add", add_numbers)

        result = await manager.run(
            pool_id="pool",
            worker_id=0,
            func_import_path_or_key="add",
            send_channel=False,
            func_args=(100, 200),
            func_kwargs={},
        )

        assert result.result == 300
        assert manager.get_num_workers("pool") == 1

In [ ]:
await test_pool_class_directly();